# Cross-Sectional Stock Return Prediction

This notebook runs the pipeline from `stock_return_project.py`.
Use the quick settings below first; scale up to more models after the pipeline is stable.


In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import stock_return_project4 as srp
srp = importlib.reload(srp)
print(srp.__file__)


/Users/hlc/Desktop/UCLA Courses/ECE C147A/Project/stock_return_project4.py


In [ ]:
# Quick run defaults.
# For a full comparison later, change MODELS to ["RNN", "LSTM", "GRU", "TRANSFORMER"].
START_DATE = "2015-01-01"
END_DATE = "2025-01-01"
LOOKBACK = 60
HORIZON = 5
MODELS = ["RNN", "LSTM", "GRU", "TRANSFORMER"]

# Universe filter: keep stocks that cover nearly the full sample window.
START_BUFFER_DAYS = 5
END_BUFFER_DAYS = 5
MIN_COVERAGE_RATIO = 0.95


In [4]:
raw_prices = srp.download_price_history(start=START_DATE, end=END_DATE)

flat_prices = srp.flatten_price_frame(raw_prices).dropna(subset=["Close"]).copy()
all_dates = pd.Index(sorted(pd.to_datetime(flat_prices["Date"].unique())))
date_to_pos = {date: idx for idx, date in enumerate(all_dates)}

coverage = (
    flat_prices.groupby("Ticker")
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        n_dates=("Date", "nunique"),
    )
    .reset_index()
)
coverage["first_pos"] = pd.to_datetime(coverage["first_date"]).map(date_to_pos)
coverage["last_pos"] = pd.to_datetime(coverage["last_date"]).map(date_to_pos)
coverage["coverage_ratio"] = coverage["n_dates"] / len(all_dates)

eligible_tickers = coverage.loc[
    (coverage["first_pos"] <= START_BUFFER_DAYS)
    & (coverage["last_pos"] >= len(all_dates) - 1 - END_BUFFER_DAYS)
    & (coverage["coverage_ratio"] >= MIN_COVERAGE_RATIO),
    "Ticker",
].tolist()

filtered_prices = raw_prices.loc[
    :,
    raw_prices.columns.get_level_values(0).isin(eligible_tickers),
]

print(
    f"Kept {len(eligible_tickers)} / {coverage.shape[0]} tickers "
    f"with >= {MIN_COVERAGE_RATIO:.0%} coverage and near-full sample history."
)
print(
    "Dropped examples:",
    coverage.loc[~coverage["Ticker"].isin(eligible_tickers), "Ticker"].head(10).tolist(),
)

experiment_data = srp.prepare_experiment_data(
    filtered_prices,
    horizon=HORIZON,
    lookback=LOOKBACK,
    train_size=0.7,
    val_size=0.15,
)
for split_name, split_df in experiment_data.splits.items():
    print(split_name, split_df["Date"].min(), split_df["Date"].max(), len(split_df))


Kept 463 / 501 tickers with >= 95% coverage and near-full sample history.
Dropped examples: ['ABNB', 'APP', 'CARR', 'CEG', 'COIN', 'CRWD', 'CTVA', 'CVNA', 'DASH', 'DDOG']
train 2015-02-02 00:00:00 2021-12-31 00:00:00 806859
val 2022-01-03 00:00:00 2023-06-29 00:00:00 173162
test 2023-06-30 00:00:00 2024-12-23 00:00:00 173162


In [ ]:
train_config = srp.TrainConfig(
    batch_size=256,
    hidden_dim=64,
    num_layers=2,
    dropout=0.1,
    num_heads=4,
    max_epochs=20,
    patience=5,
)

results = srp.run_model_suite(
    experiment_data,
    model_names=MODELS,
    train_config=train_config,
)


In [ ]:
summary = srp.build_summary_frame(results)
summary


# Result Visualizations

Plots for loss curves, IC/RankIC, portfolio performance, and model comparison.


In [ ]:
best_model = summary.iloc[0]["model"]
print("Visualizing model:", best_model)
srp.plot_training_histories(results)


In [ ]:
srp.plot_ic_series(results, best_model, split="test", rolling_window=20)


In [ ]:
srp.plot_portfolio_curve(results, best_model)
srp.plot_summary_bars(summary)


# Prediction diagnostics

Inspect raw predictions, sign accuracy, and the prediction-vs-target scatter plot.


In [ ]:
model_name = best_model
pred = results[model_name]["test_predictions"].copy()
pred.head()


In [ ]:
pred["pred_sign"] = (pred["prediction"] > 0).astype(int)
pred["true_sign"] = (pred["target"] > 0).astype(int)

sign_accuracy = (pred["pred_sign"] == pred["true_sign"]).mean()
daily_sign_accuracy = pred.groupby("Date").apply(
    lambda x: ((x["prediction"] > 0) == (x["target"] > 0)).mean(),
    include_groups=False,
)

print("Overall sign accuracy:", round(float(sign_accuracy), 4))
daily_sign_accuracy.describe()


In [ ]:
import matplotlib.pyplot as plt

latest_day = pred["Date"].max()
latest_slice = pred[pred["Date"] == latest_day].copy()
display(latest_slice.sort_values("prediction", ascending=False).head(10))

ax = latest_slice.plot.scatter(
    x="prediction",
    y="target",
    figsize=(6, 5),
    alpha=0.7,
    title=f"{model_name} prediction vs target on {latest_day.date()}"
)
ax.axhline(0.0, color="black", linewidth=1, alpha=0.6)
ax.axvline(0.0, color="black", linewidth=1, alpha=0.6)
plt.show()
